In [ ]:
def opt_shelter(population, toolbox, mate_probability, mutate_probability, ngen, stats=None, halloffame=None, log_txt_path=None):
    def binary_to_indices(binary_chrom):
        return [i for i, bit in enumerate(binary_chrom) if bit == 1]

    def _append_txt(line: str):
        if not log_txt_path:
            return
        with open(log_txt_path, "a", encoding="utf-8") as f:
            f.write(line + "\n")
            f.flush()

    logbook = tools.Logbook()
    logbook.header = ['gen', 'nevals'] + (stats.fields if stats else [])

    # 记录每一代的前三最优个体 [(gen, [idx_list...], (fitness,...)), ...]
    top_individuals_all_gens = []

    # 记录到目前为止的全局前三个个体，保存为三元组 (chromosome(binary list), fitness, generation_found)
    overall_top3 = []

    invalid_individuals = [ind for ind in population if not ind.fitness.valid]
    fitnesses = toolbox.map(toolbox.evaluate, invalid_individuals)
    for ind, fit in zip(invalid_individuals, fitnesses):
        ind.fitness.values = fit

    if halloffame is not None:
        halloffame.update(population)

    # 初始化第一代前三最优个体
    current_top3 = tools.selBest(population, 3)
    top_individuals_all_gens.append(
        (0, [binary_to_indices(ind) for ind in current_top3], [ind.fitness.values for ind in current_top3])
    )

    # 更新整体 top3
    for ind in current_top3:
        chrom = list(ind[:])
        fit = ind.fitness.values
        overall_top3.append((chrom, fit, 0))
    overall_top3.sort(key=lambda x: x[1])
    overall_top3 = overall_top3[:3]

    record = stats.compile(population) if stats else {}
    logbook.record(generation=0, nevals=len(invalid_individuals), **record)

    print(logbook.stream)
    print("第0代 当前最优的三个染色体：")
    for idx, ind in enumerate(current_top3):
        print(f"   Top{idx+1}: 染色体索引: {binary_to_indices(ind)} 适应度: {ind.fitness.values}")

    print("至目前为止整体最优的三个染色体：")
    for idx, (chrom, fit, gen_found) in enumerate(overall_top3):
        print(f"   Overall Top{idx+1}: 染色体索引: {binary_to_indices(chrom)} 适应度: {fit} 发现于第{gen_found}代")

    # 写入第 0 代（txt）
    _append_txt(
        f"gen=0 nevals={len(invalid_individuals)} stats={record} "
        f"top3={[ (binary_to_indices(ind), ind.fitness.values) for ind in current_top3 ]} "
        f"overall_top3={overall_top3}"
    )

    for gen in range(1, ngen + 1):
        offspring = toolbox.select(population, len(population))
        offspring = [toolbox.clone(ind) for ind in offspring]

        # 交叉
        for i in range(1, len(offspring), 2):
            if random.random() < mate_probability:
                offspring[i - 1], offspring[i] = toolbox.mate(offspring[i - 1], offspring[i])
                del offspring[i - 1].fitness.values, offspring[i].fitness.values

        # 变异
        for i in range(len(offspring)):
            if random.random() < mutate_probability:
                offspring[i], = toolbox.mutate(offspring[i])
                del offspring[i].fitness.values

        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = toolbox.map(toolbox.evaluate, invalid_ind)
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit

        if halloffame is not None:
            halloffame.update(offspring)

        # 当前代的前三最优个体
        current_top3 = tools.selBest(offspring, 3)
        top_individuals_all_gens.append(
            (gen, [binary_to_indices(ind) for ind in current_top3], [ind.fitness.values for ind in current_top3])
        )

        # 更新整体 top3
        for ind in current_top3:
            chrom = list(ind[:])
            fit = ind.fitness.values
            if not any(chrom == ex_chrom for (ex_chrom, _, _) in overall_top3):
                overall_top3.append((chrom, fit, gen))
        overall_top3.sort(key=lambda x: x[1])
        overall_top3 = overall_top3[:3]

        print(f"第{gen}代 当前最优的三个染色体：")
        for idx, ind in enumerate(current_top3):
            print(f"   Top{idx+1}: 染色体索引: {binary_to_indices(ind)} 适应度: {ind.fitness.values}")

        print("至目前为止整体最优的三个染色体：")
        for idx, (chrom, fit, gen_found) in enumerate(overall_top3):
            print(f"   Overall Top{idx+1}: 染色体索引: {binary_to_indices(chrom)} 适应度: {fit} 发现于第{gen_found}代")

        population[:] = offspring

        record = stats.compile(population) if stats else {}
        logbook.record(gen=gen, nevals=len(invalid_ind), **record)
        print(logbook.stream)


        _append_txt(
            f"gen={gen} nevals={len(invalid_ind)} stats={record} "
            f"top3={[ (binary_to_indices(ind), ind.fitness.values) for ind in current_top3 ]} "
            f"overall_top3={overall_top3}"
        )

    return {
        "per_gen_top3": top_individuals_all_gens,
        "overall_top3": overall_top3,
        "logbook": list(logbook),
        "log_txt_path": log_txt_path,
    }


In [ ]:
import datetime
import uuid
from pathlib import Path

# 每次运行创建一个独立实验目录
run_dir = Path("experiments") / "ga_shelter" / (datetime.datetime.now().strftime("%Y%m%d-%H%M%S") + "_" + uuid.uuid4().hex[:8])
run_dir.mkdir(parents=True, exist_ok=True)
log_txt_path = run_dir / "gen_log.txt"

# 记录一行 header（可选）
with open(log_txt_path, "a", encoding="utf-8") as f:
    f.write(f"0/1 encoding")
    f.write(f"# GA run_dir={run_dir}\n")
    f.write(f"# mate=0.4 mutate=0.2 ngen=200 pop_n={len(population)} tournsize=3\n")

results = opt_shelter(population, toolbox, 0.4, 0.2, 200, stats=stats, halloffame=hof, log_txt_path=str(log_txt_path))
print("✅ 每代日志已保存到:", log_txt_path)
print("✅ 最终 overall_top3:", results.get("overall_top3"))
